# Agrupamento de documentos

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_09/03_agrupamento_de_documentos.ipynb)

Representaremos excertos com TF-IDF e distância cosseno implícita na normalização.
K-means ainda agrupa vetores; interpretar exige examinar termos característicos e
documentos, não apenas projetar pontos coloridos.

![Textos tornam-se vetores, recebem grupos e retornam à leitura.](imagens/03_textos_clusters.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_09'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('sklearn', 'scikit-learn>=1.5,<2')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
documentos=pd.read_csv("dados/documentos_temas.csv")
vetorizador=TfidfVectorizer(min_df=2,ngram_range=(1,2),stop_words=None)
Xtexto=vetorizador.fit_transform(documentos["texto"])
modelo_texto=KMeans(n_clusters=3,random_state=42,n_init=20)
documentos["cluster_texto"]=modelo_texto.fit_predict(Xtexto)
silhueta_texto=silhouette_score(Xtexto,documentos["cluster_texto"],metric="cosine")
print(f"Silhueta cosseno: {silhueta_texto:.3f}")

## 2. Termos e documentos representativos

Os termos mais próximos do centro ajudam a descrever o padrão lexical. O documento
mais próximo do centro é um exemplar matemático, não necessariamente o mais
importante historicamente. Documentos distantes podem ser mistos ou revelar falha
da representação.

In [ ]:
termos=vetorizador.get_feature_names_out(); centros=modelo_texto.cluster_centers_
termos_cluster=[]
for c in range(3):
    top=centros[c].argsort()[-8:][::-1]
    termos_cluster.append({"cluster":c,"termos":"; ".join(termos[top])})
display(pd.DataFrame(termos_cluster))

distancias=modelo_texto.transform(Xtexto)
documentos["distancia_ao_centro"]=distancias[np.arange(len(documentos)),documentos["cluster_texto"]]
amostra_casos=pd.concat([g.nsmallest(2,"distancia_ao_centro") for _,g in documentos.groupby("cluster_texto")])
display(amostra_casos[["id_documento","periodo","cluster_texto","distancia_ao_centro","texto"]])

### U09-A04 — Atividade — do vetor ao documento

Para cada cluster, proponha um nome descritivo apoiado em termos e dois documentos.
Depois leia dois documentos mais distantes do próprio centro e registre se são
híbridos, ruidosos ou mal representados. Compare a distribuição por período sem
concluir automaticamente que o tema “cresceu”.

**Dinâmica (30 minutos):** duplas; uma nomeia pelos termos, outra tenta refutar o
nome pelos casos. A versão final deve registrar evidência favorável e exceção.

**Resposta:** escreva aqui.

## Passagem

Clusters atribuem um grupo predominante. A modelagem de tópicos permite que cada
documento seja uma mistura, mas os componentes continuam sendo padrões estatísticos.